# Méta-analyse de la recherche sur les grands modèles de langage (LLM)
### Thème : de l'instruction tuning à l'alignement — méthodes, coûts et limites

**Mini-projet 1 — Deep Learning · 5 articles analysés**

---
**Sommaire**
1. Introduction
2. Résumés des articles
3. Analyse comparative
4. Perspectives et réflexions
5. Conclusion
6. Références et liens

## 1. Introduction

Les grands modèles de langage (LLM) sont des réseaux de neurones de type Transformer, entraînés par pré-entraînement auto-supervisé sur d'immenses corpus textuels avec un objectif de prédiction du token suivant. Ce pré-entraînement leur confère de larges connaissances et des capacités de raisonnement, mais ne les rend pas pour autant utiles ni sûrs : un modèle qui prédit bien la suite d'un texte ne répond pas nécessairement à l'intention de l'utilisateur. C'est ce décalage — souvent appelé **problème d'alignement** — qui structure une grande partie de la recherche récente.

L'objectif de cette méta-analyse n'est pas de résumer cinq articles isolément, mais de reconstituer la **trajectoire méthodologique** qui va de la simple généralisation par instructions jusqu'aux méthodes d'alignement modernes, en intégrant la contrainte de coût qui pèse sur toutes ces techniques. Le fil conducteur retenu est le suivant : *comment transformer un modèle pré-entraîné en assistant qui suit des instructions, de façon utile, honnête et inoffensive, et à quel prix ?*

Les cinq articles sélectionnés se répondent explicitement les uns aux autres :

- **FLAN** (Wei et al., 2022) montre que l'instruction tuning supervisé suffit à débloquer la généralisation zero-shot.
- **InstructGPT** (Ouyang et al., 2022) montre que le signal supervisé ne suffit plus et introduit le RLHF comme standard.
- **Constitutional AI** (Bai et al., 2022) remplace une partie du feedback humain par du feedback issu du modèle lui-même, guidé par des principes écrits.
- **DPO** (Rafailov et al., 2023) démontre que l'étape d'apprentissage par renforcement peut être supprimée sans perte de qualité.
- **LoRA** (Hu et al., 2022) est l'axe transversal : il rend toutes ces phases de fine-tuning matériellement réalisables.

Les trois premiers articles définissent le **quoi** de l'alignement, les deux derniers le **comment** à moindre coût.

### Articles retenus et lieux de publication

| # | Titre abrégé | Auteurs (1er) | Année | Lieu de publication |
|---|---|---|---|---|
| A1 | Finetuned Language Models Are Zero-Shot Learners (FLAN) | Wei et al. | 2022 | ICLR 2022 |
| A2 | Training Language Models to Follow Instructions with Human Feedback (InstructGPT) | Ouyang et al. | 2022 | NeurIPS 2022 |
| A3 | Constitutional AI: Harmlessness from AI Feedback | Bai et al. | 2022 | arXiv:2212.08073 (preprint, Anthropic) |
| A4 | Direct Preference Optimization: Your Language Model is Secretly a Reward Model | Rafailov et al. | 2023 | NeurIPS 2023 |
| A5 | LoRA: Low-Rank Adaptation of Large Language Models | Hu et al. | 2022 | ICLR 2022 |

## 2. Résumés des articles

### A1 — Wei, J., Bosma, M., Zhao, V., Guu, K., Yu, A. W., Lester, B., Du, N., Dai, A. M., & Le, Q. V. (2022). *Finetuned Language Models Are Zero-Shot Learners*. ICLR 2022.

- **Problème de recherche :** le pré-entraînement produit des modèles performants en few-shot mais médiocres en zero-shot, car le format d'une instruction en langage naturel ressemble peu aux données de pré-entraînement. Comment obtenir une généralisation à des tâches jamais vues, sans exemples ?
- **Solution proposée :** l'*instruction tuning*. Les auteurs reformulent plus de 60 jeux de données NLP existants sous forme d'instructions en langage naturel (plusieurs gabarits par tâche), regroupent ces tâches en clusters, puis fine-tunent le modèle sur tous les clusters sauf celui de l'évaluation. La tâche testée n'a donc jamais été vue.
- **Principaux résultats :** le modèle obtenu, FLAN (137 milliards de paramètres, décodeur dense LaMDA-PT), surpasse GPT-3 175B en zero-shot sur une large majorité des jeux de données évalués, et le dépasse même parfois en few-shot. Résultat crucial : le bénéfice **n'apparaît qu'au-delà d'une certaine échelle** — sur des modèles plus petits, l'instruction tuning dégrade la performance zero-shot, car la capacité du modèle est consommée par les tâches d'entraînement.
- **Jeux de données :** plus de 60 jeux de données publics (NLI, QA, traduction, commonsense, analyse de sentiments…) regroupés en clusters de tâches.
- **Architecture :** Transformer décodeur uniquement, 137B paramètres, initialisé depuis LaMDA-PT.
- **Métriques :** exactitude zero-shot par cluster de tâches, comparaison directe avec GPT-3 zero-shot et few-shot, plus une évaluation humaine sur les tâches génératives.

### A2 — Ouyang, L., Wu, J., Jiang, X., Almeida, D., Wainwright, C., Mishkin, P., et al. (2022). *Training Language Models to Follow Instructions with Human Feedback*. NeurIPS 2022, vol. 35, pp. 27730–27744.

- **Problème de recherche :** agrandir un modèle ne le rend pas plus aligné sur l'intention de l'utilisateur : GPT-3 produit des sorties non véridiques, toxiques ou simplement inutiles. Le décalage porte autant sur les intentions explicites (suivre la consigne) que sur les intentions implicites (véracité, innocuité).
- **Solution proposée :** un pipeline **RLHF en trois étapes**. (1) *SFT* : fine-tuning supervisé de GPT-3 sur des démonstrations écrites par des annotateurs humains. (2) *Reward Model* : un modèle de récompense est entraîné sur des classements humains de plusieurs sorties pour un même prompt. (3) *RL* : le modèle SFT est optimisé contre ce modèle de récompense via PPO, avec une pénalité KL qui empêche la dérive vis-à-vis du modèle de référence.
- **Principaux résultats :** en évaluation humaine sur la distribution de prompts de l'API OpenAI, les sorties d'InstructGPT à **1,3 milliard** de paramètres sont préférées à celles de GPT-3 à **175 milliards** — soit 100 fois moins de paramètres. InstructGPT améliore la véracité et réduit la génération toxique, avec une régression minimale sur les benchmarks NLP publics. Les auteurs reconnaissent que le modèle commet encore des erreurs simples.
- **Jeux de données :** prompts soumis via l'API OpenAI + prompts écrits par les annotateurs ; jeux SFT, de comparaisons et de prompts RL ; évaluations sur TruthfulQA et RealToxicityPrompts.
- **Architecture :** GPT-3 (1,3B / 6B / 175B), reward model initialisé depuis un modèle de plus petite taille, optimisation PPO.
- **Métriques :** taux de préférence humaine (métrique principale), TruthfulQA, toxicité, hallucination, régression sur benchmarks publics.

### A3 — Bai, Y., Kadavath, S., Kundu, S., Askell, A., Kernion, J., Jones, A., et al. (2022). *Constitutional AI: Harmlessness from AI Feedback*. arXiv:2212.08073, Anthropic.

- **Problème de recherche :** le RLHF exige des annotateurs humains qui lisent des contenus nocifs, ce qui coûte cher, passe mal à l'échelle et n'est pas reproductible. Par ailleurs, les modèles entraînés à l'innocuité deviennent **évasifs** : ils refusent sans expliquer.
- **Solution proposée :** *Constitutional AI*, en deux phases. **Phase SL** : le modèle génère une réponse à un prompt nocif, s'autocritique au regard d'un principe tiré d'une constitution écrite par des humains, révise sa réponse, et le modèle est fine-tuné sur les réponses révisées. **Phase RL** : le modèle génère deux réponses, un modèle évaluateur choisit la meilleure selon un principe constitutionnel, ce qui produit un jeu de préférences générées par IA (**RLAIF**) mélangé aux préférences humaines d'utilité.
- **Principaux résultats :** les auteurs obtiennent un assistant à la fois inoffensif et **non évasif**, qui explique ses objections face à une requête nocive plutôt que de refuser sèchement. La seule supervision humaine réside dans la liste des principes. Le raisonnement de type chaîne de pensée améliore la qualité et la transparence des jugements de l'IA, et la capacité de l'IA à identifier les préjudices croît avec la capacité du modèle.
- **Jeux de données :** prompts nocifs issus des campagnes de red teaming, jeu de préférences HH (helpful/harmless) ; les labels d'innocuité sont générés par l'IA, ceux d'utilité restent humains.
- **Architecture :** assistant Anthropic entraîné par RLHF, modèle de préférence hybride humain/IA.
- **Métriques :** scores Elo d'utilité et d'innocuité jugés par des humains, taux d'évasivité, accord entre le modèle de préférence IA et les labels humains.

### A4 — Rafailov, R., Sharma, A., Mitchell, E., Ermon, S., Manning, C. D., & Finn, C. (2023). *Direct Preference Optimization: Your Language Model is Secretly a Reward Model*. NeurIPS 2023 (arXiv:2305.18290).

- **Problème de recherche :** le RLHF est une procédure complexe et souvent instable : il faut d'abord ajuster un modèle de récompense, puis optimiser la politique par apprentissage par renforcement sans trop s'éloigner du modèle initial. Cette boucle est coûteuse en mémoire, sensible aux hyperparamètres et difficile à reproduire.
- **Solution proposée :** les auteurs introduisent une nouvelle paramétrisation du modèle de récompense qui permet d'extraire la politique optimale **sous forme close**. En substituant cet optimum dans la perte de préférence de Bradley-Terry, on obtient une simple **perte de classification supervisée** sur des paires de préférences. Le modèle de récompense explicite et la boucle PPO disparaissent : il ne reste qu'une politique et un modèle de référence gelé.
- **Principaux résultats :** DPO est stable, sans échantillonnage pendant l'entraînement et sans réglage lourd d'hyperparamètres. Il égale ou dépasse le RLHF par PPO sur le contrôle du sentiment, la qualité du résumé et le dialogue mono-tour, tout en étant nettement plus léger à implémenter.
- **Jeux de données :** IMDb (contrôle du sentiment), TL;DR Reddit (résumé), Anthropic HH (dialogue utile/inoffensif).
- **Architecture :** GPT-2 et Pythia (jusqu'à 6B) ; deux modèles en mémoire, politique et référence gelée.
- **Métriques :** frontière récompense/divergence KL, taux de victoire évalué par GPT-4 face aux réponses de référence.

### A5 — Hu, E. J., Shen, Y., Wallis, P., Allen-Zhu, Z., Li, Y., Wang, S., Wang, L., & Chen, W. (2022). *LoRA: Low-Rank Adaptation of Large Language Models*. ICLR 2022.

- **Problème de recherche :** le fine-tuning complet d'un modèle de 175 milliards de paramètres exige de stocker et de mettre à jour l'intégralité des poids, ce qui est prohibitif dès qu'on veut une variante par tâche ou par client.
- **Solution proposée :** geler les poids pré-entraînés et injecter, dans chaque couche du Transformer, deux matrices de rang faible `A` et `B` dont le produit `BA` approxime la mise à jour. Seules `A` et `B` sont entraînées. À l'inférence, `BA` est fusionné dans les poids, donc la **latence supplémentaire est nulle**.
- **Principaux résultats :** sur GPT-3 175B, LoRA réduit le nombre de paramètres entraînables d'environ **10 000 fois** et l'empreinte mémoire GPU d'environ **3 fois**, tout en égalant ou dépassant le fine-tuning complet sur RoBERTa, DeBERTa, GPT-2 et GPT-3. Une étude du rang montre qu'un rang très faible (r = 1 à 4) suffit souvent, ce qui suggère que l'adaptation à une tâche vit dans un sous-espace de dimension réduite.
- **Jeux de données :** GLUE, WikiSQL, MNLI, SAMSum, E2E NLG.
- **Architecture :** RoBERTa, DeBERTa, GPT-2, GPT-3 175B ; adaptateurs de rang faible sur les matrices d'attention.
- **Métriques :** exactitude ou score par tâche, nombre de paramètres entraînables, mémoire, latence d'inférence, comparaison avec les *adapters* et le *prefix tuning*.

## 3. Analyse comparative

Le tableau ci-dessous confronte les cinq articles sur les dimensions structurantes. Une lecture en colonnes révèle une division du travail nette : **A1–A4 traitent de l'objectif d'apprentissage** (quel signal apprendre ?), tandis que **A5 traite de la paramétrisation** (comment l'apprendre à moindre coût ?). C'est pourquoi LoRA est *orthogonal* aux quatre autres et peut être combiné avec chacun d'eux.

### 3.1 Objectifs, architectures et stratégies d'entraînement

| Critère | A1 — FLAN | A2 — InstructGPT | A3 — Constitutional AI | A4 — DPO | A5 — LoRA |
|---|---|---|---|---|---|
| **Domaine du problème** | Généralisation zero-shot | Alignement sur l'intention | Innocuité et scalabilité de la supervision | Simplicité et stabilité de l'alignement | Coût du fine-tuning |
| **Innovation clé** | Reformuler 60+ tâches NLP en instructions | Pipeline SFT + reward model + PPO | Autocritique et révision guidées par une constitution ; RLAIF | Solution close de l'objectif RLHF ; suppression du RL | Décomposition de rang faible des mises à jour de poids |
| **Signal de supervision** | Démonstrations humaines existantes | Démonstrations + classements humains | Principes écrits + préférences générées par IA | Paires de préférences (humaines ou IA) | Indifférent (agnostique à l'objectif) |
| **Stratégie d'entraînement** | SFT multi-tâches | SFT → RM → PPO (3 étapes) | SL par révision → RLAIF (2 phases) | Perte de classification supervisée (1 étape) | Poids gelés + adaptateurs entraînables |
| **Modèle de récompense** | Aucun | Explicite | Explicite (hybride humain/IA) | Implicite (dans la politique) | Sans objet |
| **Benchmarks** | Clusters de tâches non vues, comparaison GPT-3 | Préférence humaine, TruthfulQA, toxicité | Elo utilité/innocuité, red teaming | IMDb, TL;DR, Anthropic HH | GLUE, WikiSQL, MNLI, E2E |
| **Coût de calcul** | Élevé (fine-tuning complet 137B) | Très élevé (4 modèles en jeu) | Élevé, mais peu de labels humains | Modéré (2 modèles en mémoire) | Très faible (~0,01 % des paramètres) |

### 3.2 Points forts, limites et reproductibilité

| Article | Points forts | Limites | Reproductibilité |
|---|---|---|---|
| **A1 — FLAN** | Idée simple, gain zero-shot massif, aucun mécanisme nouveau à implémenter. | Ne fonctionne qu'au-delà d'une certaine échelle ; n'optimise ni la véracité ni l'innocuité ; dépend de jeux de données NLP académiques peu représentatifs des vraies requêtes. | Bonne : jeux de données publics, gabarits d'instructions publiés ; mais LaMDA-PT 137B n'est pas ouvert. |
| **A2 — InstructGPT** | Devient le standard de facto ; démontre qu'aligner vaut mieux qu'agrandir ; évaluation humaine rigoureuse. | Coûteux ; PPO instable ; « taxe d'alignement » sur les benchmarks publics ; les préférences reflètent celles d'un petit groupe d'annotateurs anglophones ; risque de *reward hacking*. | Faible : modèles fermés, données d'annotation propriétaires, prompts issus de l'API non publics. |
| **A3 — Constitutional AI** | Réduit fortement le besoin de labels humains ; supervision explicite et auditable ; modèle non évasif. | La constitution est elle-même un choix de valeurs non neutre et non légitimé ; le modèle juge selon ses propres biais ; risque de dérive si l'évaluateur et l'évalué partagent les mêmes angles morts. | Moyenne : méthode et principes décrits en détail, mais modèles et données internes. |
| **A4 — DPO** | Élimine le RL ; stable, léger, une seule perte ; reproduit sur des modèles ouverts. | Sensible à la qualité et à la couverture du jeu de préférences ; pas d'exploration en ligne ; biais de longueur (préférence pour les réponses longues) ; performances hors distribution moins étudiées. | Excellente : GPT-2 et Pythia, jeux publics, implémentation de référence largement adoptée. |
| **A5 — LoRA** | Aucune latence ajoutée à l'inférence ; adaptateurs échangeables ; démocratise le fine-tuning. | Le choix du rang `r` et des matrices ciblées reste empirique ; peut sous-performer sur des changements de domaine majeurs par rapport au fine-tuning complet. | Excellente : code libéré, méthode indépendante du modèle, reproduite sur des architectures ouvertes. |

### 3.3 Visualisation complémentaire (facultatif)

La cellule ci-dessous positionne les cinq articles sur deux axes : **complexité du pipeline d'entraînement** (nombre d'étapes distinctes) et **dépendance à l'annotation humaine**. Elle rend visible la trajectoire de désescalade décrite en section 4.

In [ ]:
import matplotlib.pyplot as plt

# Scores qualitatifs (1 = faible, 5 = élevé), attribués à partir de la lecture des articles
articles = {
    #                     complexite_pipeline, dependance_annotation_humaine, cout_calcul (taille du point)
    "A1 FLAN":              (2, 3, 300),
    "A2 InstructGPT":       (5, 5, 500),
    "A3 Constitutional AI": (4, 2, 400),
    "A4 DPO":               (2, 4, 220),
    "A5 LoRA":              (1, 1, 90),
}

fig, ax = plt.subplots(figsize=(8, 5.5))
for nom, (x, y, taille) in articles.items():
    ax.scatter(x, y, s=taille, alpha=0.55, edgecolors="black", zorder=3)
    ax.annotate(nom, (x, y), textcoords="offset points", xytext=(0, 16),
                ha="center", fontsize=9, weight="bold")

# Trajectoire chronologique de desescalade : FLAN -> InstructGPT -> CAI -> DPO
trajet = ["A1 FLAN", "A2 InstructGPT", "A3 Constitutional AI", "A4 DPO"]
xs = [articles[a][0] for a in trajet]
ys = [articles[a][1] for a in trajet]
ax.plot(xs, ys, linestyle="--", linewidth=1.2, color="grey", zorder=1)

ax.set_xlabel("Complexite du pipeline d'entrainement (nb. d'etapes distinctes)")
ax.set_ylabel("Dependance a l'annotation humaine")
ax.set_title("Trajectoire de desescalade methodologique\n(taille du point = cout de calcul relatif)")
ax.set_xlim(0, 6); ax.set_ylim(0, 6)
ax.grid(alpha=0.25, zorder=0)
plt.tight_layout()
plt.show()

## 4. Perspectives et réflexions

### 4.1 Tendances qui se dégagent

Quatre schémas convergents ressortent de la lecture croisée.

1. **Le passage de l'échelle à l'alignement.** FLAN et InstructGPT établissent tous deux, indépendamment, qu'un modèle plus petit mais mieux entraîné bat un modèle plus gros : un InstructGPT de 1,3B est préféré à un GPT-3 de 175B. L'axe de progrès s'est déplacé du nombre de paramètres vers la qualité de la phase de post-entraînement.
2. **La simplification progressive du signal.** La trajectoire A1 → A2 → A4 est une trajectoire de *désescalade méthodologique* : d'abord on ajoute des étapes (SFT, puis modèle de récompense, puis PPO), puis on comprend qu'on peut les retirer. DPO montre que la complexité du RLHF était en partie un artefact de formulation, pas une nécessité mathématique.
3. **L'externalisation de la supervision vers le modèle lui-même.** Constitutional AI amorce un mouvement où le goulot d'étranglement — l'annotateur humain — est remplacé par un modèle guidé par des règles écrites. La supervision devient un *texte versionnable* plutôt qu'un flux d'annotations.
4. **La séparation de l'objectif et du support.** LoRA rend explicite que la question « quoi apprendre » (A1–A4) est indépendante de la question « où stocker ce qu'on apprend ». Le fine-tuning devient modulaire : un modèle de base, N adaptateurs.

### 4.2 Approches les plus prometteuses

**DPO** paraît l'innovation la plus solide sur le plan scientifique, car elle ne fait pas qu'améliorer un résultat : elle *explique pourquoi* une étape entière était superflue, en montrant que la politique encode déjà implicitement une récompense. C'est une simplification théorique, pas une astuce d'ingénierie, ce qui explique son adoption rapide.

**Constitutional AI** est l'approche la plus prometteuse sur le plan de la gouvernance : rendre les valeurs d'un modèle lisibles dans un document permet de les critiquer, de les faire évoluer et de les auditer, ce qu'un jeu d'annotations opaque ne permet pas.

La combinaison naturelle — *engendrer des préférences par constitution, puis les consommer par DPO, le tout sur des adaptateurs LoRA* — constitue aujourd'hui le pipeline d'alignement le plus accessible, et c'est effectivement celui qu'ont adopté la plupart des modèles ouverts récents.

### 4.3 Limites communément reconnues

- **Dépendance à la qualité des préférences.** A2, A3 et A4 partagent tous la même hypothèse fragile : le classement fourni (humain ou IA) reflète la « bonne » réponse. Or les annotateurs privilégient les réponses longues, confiantes et bien formatées, ce qui induit un **biais de longueur** et de **sycophantie** que ni PPO ni DPO ne corrigent.
- **Le reward hacking et la loi de Goodhart.** Dès qu'un proxy de récompense est optimisé, le modèle apprend à maximiser le proxy plutôt que l'objectif visé. La pénalité KL d'InstructGPT n'est qu'un pansement.
- **La taxe d'alignement.** InstructGPT constate une dégradation sur les benchmarks NLP publics, et FLAN une dégradation en dessous d'un certain seuil de taille : aligner consomme de la capacité.
- **La reproductibilité asymétrique.** Les articles industriels les plus influents (A2, A3) sont les moins reproductibles : modèles fermés, données d'annotation propriétaires. Les articles académiques (A4, A5) sont pleinement reproductibles mais valident leurs *claims* sur des modèles bien plus petits. Le champ progresse donc sur des preuves qui ne sont **jamais à la fois grandes et vérifiables**.
- **Le déficit d'évaluation.** Aucun de ces articles ne dispose d'une mesure satisfaisante de la véracité ou de l'innocuité : on mesure la préférence humaine, qui est une approximation, ou le jugement de GPT-4, qui est une approximation d'une approximation.

### 4.4 Orientations futures

- **Évaluations robustes et hors distribution**, moins dépendantes du jugement de préférence, et capables de détecter la sycophantie et le biais de longueur plutôt que de les récompenser.
- **Alignement à supervision vérifiable**, où le signal n'est plus une préférence subjective mais un résultat contrôlable (test unitaire, preuve formelle, exécution de code) — une direction déjà visible dans les modèles de raisonnement récents.
- **Constitutions pluralistes et légitimées**, élaborées par des procédés délibératifs plutôt qu'écrites par une entreprise, pour répondre à la principale objection adressée à A3.
- **Théorie unifiée des pertes de préférence** : DPO a ouvert une famille (IPO, KTO, ORPO, SimPO) dont les compromis exacts restent à caractériser.
- **Alignement modulaire et composable** : à quelles conditions peut-on additionner ou permuter des adaptateurs LoRA porteurs de comportements distincts sans interférence ?

## 5. Conclusion

Cette méta-analyse a suivi une trajectoire cohérente en cinq articles. **FLAN** a établi qu'un fine-tuning supervisé bien formaté suffisait à débloquer la généralisation zero-shot, à condition de disposer d'une échelle suffisante. **InstructGPT** a montré les limites de ce signal supervisé et imposé le RLHF comme standard, en démontrant qu'un modèle cent fois plus petit mais aligné était préféré à un modèle brut. **Constitutional AI** a attaqué le goulot d'étranglement humain de ce pipeline en délégant une partie du jugement au modèle, sous la contrainte de principes écrits. **DPO** a ensuite démontré que la machinerie du RL était en grande partie évitable, en dérivant une perte supervisée équivalente. **LoRA**, enfin, a rendu l'ensemble de ces phases accessibles hors des très grands laboratoires.

Le champ a donc évolué d'une **logique d'accumulation** — plus de paramètres, plus d'étapes, plus d'annotateurs — vers une **logique de simplification et d'explicitation**. Chaque article postérieur retire quelque chose au précédent : DPO retire le modèle de récompense et la boucle RL, Constitutional AI retire l'annotateur humain, LoRA retire l'essentiel des paramètres entraînables.

Ce mouvement est sain, mais il laisse intact le problème de fond que ces cinq travaux contournent plus qu'ils ne le résolvent : **nous ne savons toujours pas définir ce que nous voulons qu'un modèle fasse autrement que par la préférence d'un juge** — humain ou artificiel — dont les biais sont eux-mêmes mal caractérisés. La prochaine avancée décisive portera vraisemblablement moins sur l'optimiseur que sur la nature du signal optimisé.

## 6. Références et liens

1. Wei, J., Bosma, M., Zhao, V., et al. (2022). *Finetuned Language Models Are Zero-Shot Learners*. ICLR 2022 — https://arxiv.org/abs/2109.01652
2. Ouyang, L., Wu, J., Jiang, X., et al. (2022). *Training Language Models to Follow Instructions with Human Feedback*. NeurIPS 2022 — https://arxiv.org/abs/2203.02155
3. Bai, Y., Kadavath, S., Kundu, S., et al. (2022). *Constitutional AI: Harmlessness from AI Feedback*. arXiv — https://arxiv.org/abs/2212.08073
4. Rafailov, R., Sharma, A., Mitchell, E., et al. (2023). *Direct Preference Optimization: Your Language Model is Secretly a Reward Model*. NeurIPS 2023 — https://arxiv.org/abs/2305.18290
5. Hu, E. J., Shen, Y., Wallis, P., et al. (2022). *LoRA: Low-Rank Adaptation of Large Language Models*. ICLR 2022 — https://arxiv.org/abs/2106.09685

**Références secondaires citées en section 4**

6. Schulman, J., et al. (2017). *Proximal Policy Optimization Algorithms* — https://arxiv.org/abs/1707.06347
7. Casper, S., et al. (2023). *Open Problems and Fundamental Limitations of RLHF* — https://arxiv.org/abs/2307.15217